# Loading Data To/From S3

Download the dataset from the kaggle link in the README and store it in your local system. 
Provide that path to load the data and upload to s3.

This module covers:
1. Bucket/prefix conventions we'll use for the rest of the course
2. Uploading data to S3 two ways: the `sagemaker` SDK helper and raw `boto3`
3. Reading data back from S3 into pandas, two ways

In [ ]:
import boto3
import sagemaker


REGION = ""
BUCKET_NAME = ""
# ------------------------------------------------

boto_session = boto3.Session(region_name=REGION)
sm_session = sagemaker.Session(boto_session=boto_session)
s3 = boto_session.client("s3")

## 1. Bucket/prefix conventions

We'll use this layout for every module going forward:

```
s3://<bucket>/raw/          original, untouched data
s3://<bucket>/processed/    cleaned / feature-engineered data
s3://<bucket>/models/       training job outputs (populated automatically by Estimators later)
```

Keeping `raw/` immutable (never overwritten) means every later step can be re-run from
the same starting point — useful both for debugging and for re-recording course demos.

## 2a. Upload via the `sagemaker` SDK

`Session.upload_data()` is the SageMaker-idiomatic way to get local files into S3 — it
returns the resulting `s3://` URI directly, which is handy for wiring straight into an
Estimator's input channels in later modules.

In [ ]:
# Give the path to the local data file that you want to upload to S3
LOCAL_DATA_PATH = ""

raw_s3_uri = sm_session.upload_data(
    path=LOCAL_DATA_PATH,
    bucket=BUCKET_NAME,
    key_prefix="raw",
)
print(f"Uploaded to: {raw_s3_uri}")

## 2b. Upload via raw `boto3` (the equivalent lower-level call)

Useful to know because not everything in SageMaker goes through the SDK's convenience
methods — Processing job outputs, custom artifacts, etc. often get pushed with `boto3`
directly.

In [ ]:
### SKIP THIS FOR NOW

# s3.upload_file(
#     Filename=LOCAL_DATA_PATH,
#     Bucket=BUCKET_NAME,
#     Key="raw/sample_data_via_boto3.csv",
# )
# print(f"Uploaded to: s3://{BUCKET_NAME}/raw/sample_data_via_boto3.csv")

## 3. Reading data back from S3 into pandas

### 3a. Via `s3fs` — pandas can read an `s3://` URI directly once `s3fs` is installed

In [ ]:
#  import pandas as pd
# df = pd.read_csv(raw_s3_uri)
# print(df.shape)
# df.head()

### 3b. Via `boto3` + `io.BytesIO` — useful when you don't want the `s3fs` dependency, or
need finer control (e.g. reading only part of a large object)

In [ ]:
import io
import pandas as pd

obj = s3.get_object(Bucket=BUCKET_NAME, Key="raw/sample_data.csv")

In [ ]:
df_boto3 = pd.read_csv(io.BytesIO(obj["Body"].read()))
print(df_boto3.shape)
df_boto3.head()